### Music Data List

In [1]:
import os
from os import listdir
from os.path import isfile, join
import warnings
warnings.filterwarnings("ignore")


data_train_dir = 'dataset_after_preprocess/train'
data_train_files = []
data_train_files += [join(data_train_dir, f) for f in listdir(data_train_dir) if isfile(join(data_train_dir, f)) if '.npz' in f]
data_train_files.sort()

data_test_dir = 'dataset_after_preprocess/test'
data_test_files = []
data_test_files += [join(data_test_dir, f) for f in listdir(data_test_dir) if isfile(join(data_test_dir, f)) if '.npz' in f]
data_test_files.sort()

print(len(data_train_files))
print(len(data_test_files))
print(data_train_files)
print(data_train_files)

967
178
['dataset_after_preprocess/train/0.npz', 'dataset_after_preprocess/train/1.npz', 'dataset_after_preprocess/train/10.npz', 'dataset_after_preprocess/train/100.npz', 'dataset_after_preprocess/train/101.npz', 'dataset_after_preprocess/train/102.npz', 'dataset_after_preprocess/train/103.npz', 'dataset_after_preprocess/train/104.npz', 'dataset_after_preprocess/train/105.npz', 'dataset_after_preprocess/train/106.npz', 'dataset_after_preprocess/train/107.npz', 'dataset_after_preprocess/train/108.npz', 'dataset_after_preprocess/train/109.npz', 'dataset_after_preprocess/train/11.npz', 'dataset_after_preprocess/train/110.npz', 'dataset_after_preprocess/train/111.npz', 'dataset_after_preprocess/train/112.npz', 'dataset_after_preprocess/train/113.npz', 'dataset_after_preprocess/train/114.npz', 'dataset_after_preprocess/train/115.npz', 'dataset_after_preprocess/train/116.npz', 'dataset_after_preprocess/train/117.npz', 'dataset_after_preprocess/train/118.npz', 'dataset_after_preprocess/train

### Hyperparameters Setting

In [2]:
IntervalDim = 100
VelocityDim = 32
# CCDim = 2
NoteOnDim = 128
NoteOffDim = 128

VelocityOffset = IntervalDim
NoteOnOffset = IntervalDim + VelocityDim
NoteOffOffset = IntervalDim + VelocityDim + NoteOnDim
# CCOffset = IntervalDim + VelocityDim + NoteOnDim + NoteOffDim

EventDim = IntervalDim + VelocityDim + NoteOnDim + NoteOffDim # 388

Time = 2000 #处理的每个sample的维度
EmbeddingDim = 512 #把每个sample嵌入的维度
HeadDim = 32 #头的维度
Heads = 16 #多少个头
ContextDim = HeadDim * Heads # 512
Layers = 8 #多少个decoder

In [3]:
import numpy as np
import tensorflow as tf
from tensorflow.contrib.training import HParams

def default_hparams():
    return HParams(
        n_vocab=EventDim,
        n_ctx=ContextDim,
        n_embd=EmbeddingDim,
        n_head=Heads,
        n_layer=Layers,
        n_time=Time,
    )

hparams = default_hparams()
print(hparams)

n_vocab=388,n_ctx=512,n_embd=512,n_head=16,n_layer=8,n_time=2000


### Load data from npz and converter to token sequence

In [4]:
import numpy as np
import matplotlib.pyplot as plt
import librosa.display

def get_data(length, data_files, typ, iteration):
    if typ == 'train':
        index = np.random.randint(0, len(data_files))
    elif typ == 'test':
        index = iteration
        
    data = np.load(data_files[index])['eventlist']  
    
    # time augmentation
    data[:, 0] *= np.random.uniform(0.80, 1.20)
    
    # absolute time to relative interval
    data[1:, 0] = data[1:, 0] - data[:-1, 0]
    data[0, 0] = 0
    
    # discretize interval into IntervalDim
    data[:, 0] = np.clip(np.round(data[:, 0] * IntervalDim), 0, IntervalDim - 1)
    
    # Note augmentation
    data[:, 2] += np.random.randint(-6, 6)
    data[:, 2] = np.clip(data[:, 2], 0, NoteOnDim - 1)
    
    eventlist = []
    for d in data:
        # append interval
        interval = d[0]
        eventlist.append(interval)
    
        # note on case
        if d[1] == 1:
            velocity = (d[3] / 128) * VelocityDim + VelocityOffset
            note = d[2] + NoteOnOffset
            eventlist.append(velocity)
            eventlist.append(note)
            
        # note off case
        elif d[1] == 0:
            note = d[2] + NoteOffOffset
            eventlist.append(note)
        # CC
        elif d[1] == 2:
            event = CCOffset + d[3]
            eventlist.append(event)
            
    eventlist = np.array(eventlist).astype(np.int)
    
    if len(eventlist) > (length+1):
        start_index = np.random.randint(0, len(eventlist) - (length+1))
        eventlist = eventlist[start_index:start_index+(length+1)]
        
    # pad zeros
    if len(eventlist) < (length+1):
        pad = (length+1) - len(eventlist)
        eventlist = np.pad(eventlist, (pad, 0), 'constant')
        
    x = eventlist[:length]
    y = eventlist[1:length+1] #y就是x的下一步
    
    return x, y
    
# x, y = get_data()
# print('x shape : ', x.shape)
# print('y shape : ', y.shape)
   
    
# roll = np.zeros([len(x), EventDim])
# for t, _x in enumerate(x):
#     roll[t, _x] = 1

# plt.figure(figsize=[18, 15])
# librosa.display.specshow(roll.T)
# plt.show()

### GPT-2 source code from https://github.com/openai/gpt-2/blob/master/src/model.py

In [10]:
def shape_list(x):
    """Deal with dynamic shape in tensorflow cleanly."""
    static = x.shape.as_list()
    dynamic = tf.shape(x)
    return [dynamic[i] if s is None else s for i, s in enumerate(static)]

def softmax(x, axis=-1):
    x = x - tf.reduce_max(x, axis=axis, keepdims=True)
    ex = tf.exp(x)
    return ex / tf.reduce_sum(ex, axis=axis, keepdims=True)

def gelu(x):
    return 0.5*x*(1+tf.tanh(np.sqrt(2/np.pi)*(x+0.044715*tf.pow(x, 3))))

def norm(x, scope, *, axis=-1, epsilon=1e-5):
    """Normalize to mean = 0, std = 1, then do a diagonal affine transform."""
    with tf.variable_scope(scope):
        n_state = x.shape[-1].value
        g = tf.get_variable('g', [n_state], initializer=tf.constant_initializer(1))
        b = tf.get_variable('b', [n_state], initializer=tf.constant_initializer(0))
        u = tf.reduce_mean(x, axis=axis, keepdims=True)
        s = tf.reduce_mean(tf.square(x-u), axis=axis, keepdims=True)
        x = (x - u) * tf.rsqrt(s + epsilon)
        x = x*g + b
        return x

def split_states(x, n):
    """Reshape the last dimension of x into [n, x.shape[-1]/n]."""
    *start, m = shape_list(x)
    return tf.reshape(x, start + [n, m//n])

def merge_states(x):
    """Smash the last two dimensions of x into a single dimension."""
    *start, a, b = shape_list(x)
    return tf.reshape(x, start + [a*b])

def conv1d(x, scope, nf, *, w_init_stdev=0.02):
    with tf.variable_scope(scope):
        *start, nx = shape_list(x)
        w = tf.get_variable('w', [1, nx, nf], initializer=tf.random_normal_initializer(stddev=w_init_stdev))
        b = tf.get_variable('b', [nf], initializer=tf.constant_initializer(0))
        c = tf.reshape(tf.matmul(tf.reshape(x, [-1, nx]), tf.reshape(w, [-1, nf]))+b, start+[nf])
        return c

def attention_mask(nd, ns, *, dtype):
    """1's in the lower triangle, counting from the lower right corner.
    Same as tf.matrix_band_part(tf.ones([nd, ns]), -1, ns-nd), but doesn't produce garbage on TPUs.
    """
    i = tf.range(nd)[:,None]
    j = tf.range(ns)
    m = i >= j - ns + nd
    return tf.cast(m, dtype)

'''
MEMORY EFFICIENT IMPLEMENTATION OF RELATIVE POSITION-BASED ATTENTION
(Music Transformer, Cheng-Zhi Anna Huang et al. 2018)
'''
def attn(x, scope, n_state, *, hparams):
    assert x.shape.ndims == 3  # Should be [batch, sequence, features]
    assert n_state % hparams.n_head == 0

    def split_heads(x):
        # From [batch, sequence, features] to [batch, heads, sequence, features]
        return tf.transpose(split_states(x, hparams.n_head), [0, 2, 1, 3])

    def merge_heads(x):
        # Reverse of split_heads
        return merge_states(tf.transpose(x, [0, 2, 1, 3]))

    def mask_attn_weights(w):
        # w has shape [batch, heads, dst_sequence, src_sequence], where information flows from src to dst.
        _, _, nd, ns = shape_list(w)
        b = attention_mask(nd, ns, dtype=w.dtype)
        b = tf.reshape(b, [1, 1, nd, ns])
        w = w*b - tf.cast(1e10, w.dtype)*(1-b)
        return w
    
    def relative_attn(q):
        # q have shape [batch, heads, sequence, features]
        batch, heads, sequence, features = shape_list(q)
        E = tf.get_variable('E', [heads, sequence, features])
        # [heads, batch, sequence, features]
        q_ = tf.transpose(q, [1, 0, 2, 3])
        # [heads, batch * sequence, features]
        q_ = tf.reshape(q_, [heads, batch * sequence, features])
        # [heads, batch * sequence, sequence]
        rel = tf.matmul(q_, E, transpose_b=True)
        # [heads, batch, sequence, sequence]
        rel = tf.reshape(rel, [heads, batch, sequence, sequence])
        # [heads, batch, sequence, 1+sequence]
        rel = tf.pad(rel, ((0, 0), (0, 0), (0, 0), (1, 0)))
        # [heads, batch, sequence+1, sequence]
        rel = tf.reshape(rel, (heads, batch, sequence+1, sequence))
        # [heads, batch, sequence, sequence]
        rel = rel[:, :, 1:]
        # [batch, heads, sequence, sequence]
        rel = tf.transpose(rel, [1, 0, 2, 3])
        return rel
        
    def multihead_attn(q, k, v):
        # q, k, v have shape [batch, heads, sequence, features]
        w = tf.matmul(q, k, transpose_b=True)
        w = w + relative_attn(q)
        w = w * tf.rsqrt(tf.cast(v.shape[-1].value, w.dtype))

        w = mask_attn_weights(w)
        w = softmax(w)
        a = tf.matmul(w, v)
        return a

    with tf.variable_scope(scope):
        c = conv1d(x, 'c_attn', n_state*3)
        q, k, v = map(split_heads, tf.split(c, 3, axis=2))
        present = tf.stack([k, v], axis=1)

        a = multihead_attn(q, k, v)
        a = merge_heads(a)
        a = conv1d(a, 'c_proj', n_state)
        return a, present


def mlp(x, scope, n_state, *, hparams):
    with tf.variable_scope(scope):
        nx = x.shape[-1].value
        h = gelu(conv1d(x, 'c_fc', n_state))
        h2 = conv1d(h, 'c_proj', nx)
        return h2


def block(x, scope, *, hparams): #一个decoder块
    with tf.variable_scope(scope):
        nx = x.shape[-1].value
        a, present = attn(norm(x, 'ln_1'), 'attn', nx, hparams=hparams) #Attention
        x = x + a #残差
        m = mlp(norm(x, 'ln_2'), 'mlp', nx*4, hparams=hparams) #前馈神经网络
        x = x + m #残差
        return x, present

def expand_tile(value, size):
    """Add a new axis of given size."""
    value = tf.convert_to_tensor(value, name='value')
    ndims = value.shape.ndims
    return tf.tile(tf.expand_dims(value, axis=0), [size] + [1]*ndims)

def model(hparams, X, scope='model', reuse=False):
    with tf.variable_scope(scope, reuse=reuse):
        results = {}
        batch, sequence = shape_list(X)

        wte = tf.get_variable('wte', [hparams.n_vocab, hparams.n_embd],
                             initializer=tf.random_normal_initializer(stddev=0.02))
        h = tf.gather(wte, X)

        # Transformer
        presents = []
        for layer in range(hparams.n_layer):
            h, present = block(h, 'h%d' % layer, hparams=hparams)
            presents.append(present)
        results['present'] = tf.stack(presents, axis=1)
        h = norm(h, 'ln_f')
        
        # Linear
        nh = h.shape[-1].value
        l = mlp(norm(h, 'ln_2'), 'mlp', nh*8, hparams=hparams) 

        # Language model loss.  Do tokens <n predict token n?
        h_flat = tf.reshape(l, [batch*sequence, hparams.n_embd])
        logits = tf.matmul(h_flat, wte, transpose_b=True)
        logits = tf.reshape(logits, [batch, sequence, hparams.n_vocab])
        results['logits'] = logits
        return results

### Draw Main Graph

In [6]:
hparams = default_hparams()
print(hparams)

tf.reset_default_graph()

X = tf.placeholder(tf.int32, [None, hparams.n_time])
Y = tf.placeholder(tf.int32, [None, hparams.n_time])

X_onehot = tf.one_hot(X, axis=2, depth=hparams.n_vocab)

logits = model(hparams, X)['logits']
probs = tf.nn.softmax(logits, axis=2)
cross_entropy = tf.nn.sparse_softmax_cross_entropy_with_logits(labels=Y, logits=logits)
loss = tf.reduce_mean(cross_entropy)

temperature = 0
u = tf.random.uniform(shape=tf.shape(logits[:, -1]), minval=1e-5, maxval=1.-1e-5)
u = (logits[:, -1] - tf.log(temperature + 1e-8)) - tf.log(-tf.log(u))
sample = tf.argmax(u, axis=1)

'''
Train
'''
global_step = tf.Variable(0, name='global_step')
learning_rate = tf.Variable(1e-3, name='learning_rate')
train_step = tf.train.AdamOptimizer(learning_rate).minimize(loss, global_step)

'''
Session Open
'''

# GPU number to use
gpu_options = tf.GPUOptions(visible_device_list="0")
sess = tf.Session(config=tf.ConfigProto(gpu_options=gpu_options))

sess.run(tf.global_variables_initializer())

print('graph create')

n_vocab=388,n_ctx=512,n_embd=512,n_head=16,n_layer=8,n_time=2000
Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor
Instructions for updating:
Use tf.where in 2.0, which has the same broadcast rule as np.where
graph create


### Load model if exist

In [7]:
import tensorflow.contrib.slim as slim
from tensorflow.python import pywrap_tensorflow

load_dir = 'save/gpt2-cc-interval100-attention2000-midi'
save_dir = 'save/gpt2-cc-interval100-attention2000-midi'

def get_variables_from_checkpoint_file(file_name):
    variables = []
    reader = pywrap_tensorflow.NewCheckpointReader(file_name)

    var_to_shape_map = reader.get_variable_to_shape_map()
    for key in sorted(var_to_shape_map):
        variables.append((key, var_to_shape_map[key]))

    return variables

saver = tf.train.Saver()

if True:
    restore_file = tf.train.latest_checkpoint(load_dir)
    print(restore_file)
    if restore_file is not None:
        try:
            saver.restore(sess, restore_file)
            print("Model restored.", restore_file)
        except:
            saved_variables = get_variables_from_checkpoint_file(restore_file)
            model_variables = slim.get_variables_to_restore()
            restore_variables = []
            for model_variable in model_variables:
                for saved_variable_name, saved_variable_shape in saved_variables:
                    model_variable_name = model_variable.name.split(":")[0]
                    if saved_variable_name == model_variable_name and tuple(saved_variable_shape) == model_variable.shape:
                        restore_variables.append(model_variable)

            init_saver = tf.train.Saver(restore_variables)
            init_saver.restore(sess, restore_file)
            print("Model partially restored.")
    else:
        print('model not exist.')
        

None
model not exist.


### TensorboardX Logger

In [8]:
from tensorboardX import SummaryWriter

class Logger(SummaryWriter):
    def __init__(self, logdir):
        super(Logger, self).__init__(logdir)

    def log(self, log_string, value, iteration):
            self.add_scalar(log_string, value, iteration)
            
logger = Logger(save_dir)            

### Train Loop

In [15]:
from IPython.display import clear_output
from tqdm import tqdm_notebook as tqdm
import matplotlib.pyplot as plt
import librosa.display
from time import sleep
import time
import math

batch_size = 1

print('iteration\t', 'loss\t', 'train_perplexity\t')
while(True):
    for _ in range(100):
        _inputs = []
        _targets = []
        for _ in range(batch_size):
            while(True):
                x, y = get_data(hparams.n_time, data_train_files,'train',_)
                if(x.shape == y.shape):
                    break
                 
            _inputs.append(x)
            _targets.append(y)
        _inputs = np.stack(_inputs)
        _targets = np.stack(_targets)
#         print(_inputs.shape, _targets.shape)
        
        _, _global_step, _loss = sess.run([train_step, global_step, loss], 
                                          feed_dict={X: _inputs, 
                                                     Y: _targets,
                                                     learning_rate: 1e-3})
        
        train_perplexity = math.exp(_loss) #log perplexity和交叉熵等价
        print(str(_global_step)+'\t', str(_loss)+'\t', str(train_perplexity)+'\t')
        
        if _global_step % 10 == 0:
            logger.log('loss', _loss, _global_step)
        
        if _global_step % 1000 == 0:
            save_path = saver.save(sess, save_dir + '/checkpoint', global_step=_global_step)
            print("Model saved in path: %s" % save_path)

iteration	 loss	 train_perplexity	
17	 5.3144455	 203.25177787161246	
18	 4.6773596	 107.48588976940907	
19	 4.6612053	 105.7634812158733	
20	 4.8954415	 133.67901716791644	
21	 4.83213	 125.47793862763628	


KeyboardInterrupt: 

### Compute perplexity on test set

In [21]:
inputs = []
targets = []

for i in range(len(data_test_files)): 
    while(True):
        x_test, y_test = get_data(hparams.n_time, data_test_files,'test',i)
        if(x_test.shape == y_test.shape):
            break       
    inputs.append(x_test)
    targets.append(y_test)
inputs = np.stack(inputs)
targets = np.stack(targets)

test_loss = sess.run(loss, feed_dict={X: inputs, Y: targets})
test_perplexity = math.exp(test_loss)
test_perplexity


# #如果内存不够，用下面这个
# test_perplexity_list = []
# for i in range(len(data_test_files)): 
#     inputs = []
#     targets = []
#     while(True):
#         x_test, y_test = get_data(hparams.n_time, data_test_files,'test',i)
#         if(x_test.shape == y_test.shape):
#             break       
#     inputs.append(x_test)
#     targets.append(y_test)
#     inputs = np.stack(inputs)
#     targets = np.stack(targets)

#     test_loss = sess.run(loss, feed_dict={X: inputs, Y: targets})
#     test_perplexity = math.exp(test_loss)
#     test_perplexity_list.append(test_perplexity)

# perplexity = sum(test_perplexity_list)/len(test_perplexity_list)    
# perplexity

102.49696838768455